In [ ]:
# ============================================================================
# Imports
# ============================================================================

from pathlib import Path

import pandas as pd

from hermes.data_catalog import get_dataset
from hermes.io import download_file
from hermes.preprocessing import prepare_mayotte_employment
from hermes.integration import integrate_employment
from hermes.loaders import load_employment
from hermes.config import PREPARED_DIR
from hermes.io import save_dataframe

In [ ]:
# ============================================================================
# Get dataset
# ============================================================================

dataset = get_dataset(
    "mayotte_employment_raw"
)

print(dataset.download_path)
print(dataset.local_path)

In [ ]:
# ============================================================================
# Download dataset
# ============================================================================

archive_path = download_file(
    url=dataset.url,
    destination=dataset.download_path,
)

In [ ]:
# ============================================================================
# Inspect extracted files
# ============================================================================

for path in sorted(
    archive_path.parent.rglob("*.xls")
):
    print(path)

In [ ]:
# ============================================================================
# Locate employment workbooks
# ============================================================================

employment_files = {
    path.stem: path
    for path in archive_path.parent.glob("BTX_TD_ACT*_2017.xls")
}

employment_files

In [ ]:
# ============================================================================
# Inspect workbook sheets
# ============================================================================

for name, path in employment_files.items():
    workbook = pd.ExcelFile(
        path,
        engine="xlrd",
    )

    print(name, workbook.sheet_names)

In [ ]:
# ============================================================================
# Inspect workbook contents
# ============================================================================

for name, path in employment_files.items():

    workbook = pd.ExcelFile(
        path,
        engine="xlrd",
    )

    sheet = workbook.sheet_names[0]

    preview = pd.read_excel(
        path,
        sheet_name=sheet,
        engine="xlrd",
        header=None,
    )

    print("\n", "=" * 80)
    print(name, "-", sheet)
    print("=" * 80)

    display(
        preview.head(12)
    )

In [ ]:
# ============================================================================
# Inspect ACT1 commune table
# ============================================================================

act1_path = employment_files[
    "BTX_TD_ACT1_2017"
]

act1_raw = pd.read_excel(
    act1_path,
    sheet_name="COM",
    engine="xlrd",
    header=None,
)

act1_raw.head(20)

In [ ]:
# ============================================================================
# Inspect ACT1 variables
# ============================================================================

act1_variables = pd.read_excel(
    act1_path,
    sheet_name="Liste des variables",
    engine="xlrd",
    header=None,
)

act1_variables.head(50)

In [ ]:
act1_raw.head(15).iloc[:, :20]

In [ ]:
for name in [
    "BTX_TD_ACT2A_2017",
    "BTX_TD_ACT2B_2017",
    "BTX_TD_ACT3_2017",
    "BTX_TD_ACT4_2017",
    "BTX_TD_ACT5_2017",
]:
    path = employment_files[name]

    preview = pd.read_excel(
        path,
        sheet_name="COM",
        engine="xlrd",
        header=None,
        nrows=3,
    )

    print(name)
    print(preview.iloc[0, 0])
    print()

In [ ]:
# ============================================================================
# Load Mayotte population indicators
# ============================================================================

from pathlib import Path

population_path = Path(
    "../data/raw/rp-mayotte-2017-ind-communes.xls"
)

ind1_raw = pd.read_excel(
    population_path,
    sheet_name="IND1",
    engine="xlrd",
    header=None,
)

In [ ]:
# ============================================================================
# Prepare employment data with missing Mayotte data
# ============================================================================

mayotte_employment = prepare_mayotte_employment(
    ind1_raw,
    act1_raw,
)

# Check 17x10
print(mayotte_employment.shape)
mayotte_employment.head()

In [ ]:
# ============================================================================
# Load full employment data
# ============================================================================

employment = load_employment()

employment_integrated = integrate_employment(
    employment,
    mayotte_employment,
)

In [ ]:
# Sanity check

print(employment.shape)
print(mayotte_employment.shape)
print(employment_integrated.shape)

In [ ]:
employment_integrated[
    employment_integrated["insee_code"] == "97602"
]

In [ ]:
# ============================================================================
# Save integrated employment dataset
# ============================================================================

save_dataframe(
    employment_integrated,
    PREPARED_DIR / "employment.parquet",
)